# 02 — Continued pre-training (CPT) on CPU

**SFT** (notebook 01) teaches a model a behavior from request → answer pairs. **Continued pre-training** teaches
it *knowledge* from plain text: the same next-token objective the model was pre-trained with, on documents from
your domain, with no chat template and no answers.

The domain here is a fictional restaurant, *Trattoria Lumen* (its chef, opening hours, signature dish, ...), so
the base model cannot know any of it. The job:

1. asks the base model 8 questions about the restaurant, phrased differently from the training text
   (fill-in-the-blank), and measures its perplexity on 8 held-out sentences
2. trains a LoRA adapter on 200 short generated documents (60 steps, LoRA r=16 on attention + MLP)
3. asks the same questions and measures perplexity again

A high score after training means the model learned the facts, not the sentences: the questions never appear in
the training text. Expected: ~5 minutes, mostly job startup.

In [ ]:
%pip install -q "sagemaker==3.21.0"

In [ ]:
import sys

sys.path.insert(0, "scripts")
from sm_helpers import Lab

lab = Lab()
CAPACITY = "fallback"   # "on-demand" | "spot" | "fallback" (compared side by side in 01-sft.ipynb)

In [ ]:
import booking_data

print(booking_data.cpt_documents(n_docs=1)[0], "\n")
for key in ("chef", "closed_day"):
    print(f"{booking_data.CLOZE[key]} ...   (expected: {booking_data.FACTS[key]})")

## Train

In [ ]:
job = lab.submit("cpt", "train_cpt.py", {"steps": 60, "model_id": "Qwen/Qwen3-0.6B"}, capacity=CAPACITY)

In [ ]:
job = globals().get("job") or lab.latest("cpt")   # after a kernel restart
desc = lab.wait(job)

## Results

In [ ]:
m = lab.metrics(job)
print(f"{'':10} {'questions answered':>19} {'perplexity (held-out)':>22}")
for which in ("base", "finetuned"):
    print(f"{which:10} {m[which]['cloze_accuracy']:19.0%} {m[which]['heldout_perplexity']:22.1f}")
print()
for ex in m["examples"]:
    print(f"{ex['question']}\n   base:      {ex['base']!r}\n   finetuned: {ex['finetuned']!r}\n")

Lower perplexity means the held-out sentences are now much less surprising to the model. In practice CPT is the
first step when a model must speak a domain's language or know its facts; SFT and preference tuning then shape
how it answers.